In [18]:
from pathlib import Path
from typing import List, Tuple
import os
import csv

In [9]:
root = Path().resolve().parent.as_posix()

os.chdir(root)


In [10]:
os.getcwd()

'/Users/harendrakumar/Documents/rag-production/rag_project'

In [12]:
from src.loader import PDFLoader
from src.embeddings import EmbeddingService
from src.chunker import  DocumentChunker
from src.vector_store import FAISSVectorStore
from src.retriever import DocumentRetriever


from evaluation.config import RetrievalConfig
from evaluation.results import RetrievalResult
from evaluation.metrics import (
    calculate_f1, 
    calculate_precision_at_k,
    calculate_recall_at_k,
    get_relevant_chunk_ids
    )

from evaluation.retrieval_dataset import EVALUATION_DATASET

In [13]:
PDF_PATH = "data/documents/understanding-the-dom.pdf"

In [22]:
class RetrievalExperiment:
    def __init__(self, pdf_path=PDF_PATH):
        self.pdf_path = pdf_path
        self.loader =PDFLoader()
        self.embedding_service = EmbeddingService()


    def run_chunking_configuration(
            self,
            chunk_size: int,
            chunk_overlap: int,
            top_k_values: List[int]
            ) -> List[RetrievalResult]:
        if not top_k_values:
            raise ValueError("At least one of top_k value is required")

        for item in EVALUATION_DATASET:
            if not item["relevant_pages"]:
                raise ValueError(f"Add verified relevant pages for: {item['question']}")


        documents = self.loader.load(self.pdf_path)

        chunker = DocumentChunker(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        chunks = chunker.split(documents=documents)

        # Generate document embeddings once for this chunking step
        texts = [chunk.page_content for chunk in chunks]
        embeddings = self.embedding_service.embed_documents(texts)

        vector_store = FAISSVectorStore()
        vector_store.add(documents=chunks, embeddings=embeddings)

        retriever = DocumentRetriever(embedding_service=self.embedding_service, 
                                      vector_store=vector_store)

        results_for_configuration = []

        # Reuse the same vector index for different top_k values.

        for top_k in top_k_values:
            recalls = []
            precisions = []

            for item in EVALUATION_DATASET:
                relevant_ids = get_relevant_chunk_ids(chunks=chunks,
                                                      relevant_pages=item["relevant_pages"]
                                                    )
                retrieved = retriever.retrieve(question=item["question"], top_k=top_k)
                recalls.append(
                    calculate_recall_at_k(retrieved_chunks=retrieved, relevant_chunk_ids=relevant_ids)
                )

                precisions.append(
                    calculate_precision_at_k(retrieved, relevant_ids)
                )

            avg_recall = sum(recalls)/ len(recalls)
            avg_precision = sum(precisions)/ len(precisions)

            result = RetrievalResult(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
                top_k=top_k,
                total_chunks=len(chunks),
                recall=avg_recall,
                precision=avg_precision,
                f1=calculate_f1(avg_precision, avg_recall)
                )
            results_for_configuration.append(result)

        return results_for_configuration

In [23]:
import csv



CHUNKING_CONFIGS = [
    (500, 50),
    (700, 200),
    (1200, 150),
]

TOP_K_VALUES = [1, 3, 5, 10]


def main():
    experiment = RetrievalExperiment()
    all_results = []

    for chunk_size, chunk_overlap in CHUNKING_CONFIGS:
        print(
            f"\nTesting chunk_size={chunk_size}, "
            f"chunk_overlap={chunk_overlap}"
        )

        results = experiment.run_chunking_configuration(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            top_k_values=TOP_K_VALUES,
        )
        all_results.extend(results)

        for result in results:
            print(
                f"K={result.top_k:2} | "
                f"chunks={result.total_chunks:4} | "
                f"Recall={result.recall:.4f} | "
                f"Precision={result.precision:.4f} | "
                f"F1={result.f1:.4f}"
            )

    output_path = "evaluation_results.csv"

    with open(output_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=[
                "chunk_size",
                "chunk_overlap",
                "top_k",
                "total_chunks",
                "recall",
                "precision",
                "f1",
            ],
        )
        writer.writeheader()

        for result in all_results:
            writer.writerow({
                "chunk_size": result.chunk_size,
                "chunk_overlap": result.chunk_overlap,
                "top_k": result.top_k,
                "total_chunks": result.total_chunks,
                "recall": round(result.recall, 4),
                "precision": round(result.precision, 4),
                "f1": round(result.f1, 4),
            })

    print(f"\nResults saved to {output_path}")


if __name__ == "__main__":
    main()




Testing chunk_size=500, chunk_overlap=50
K= 1 | chunks= 222 | Recall=0.0357 | Precision=0.2500 | F1=0.0625
K= 3 | chunks= 222 | Recall=0.2905 | Precision=0.5000 | F1=0.3675
K= 5 | chunks= 222 | Recall=0.3762 | Precision=0.4000 | F1=0.3877
K=10 | chunks= 222 | Recall=0.4952 | Precision=0.2500 | F1=0.3323

Testing chunk_size=700, chunk_overlap=200
K= 1 | chunks= 173 | Recall=0.0917 | Precision=0.5000 | F1=0.1549
K= 3 | chunks= 173 | Recall=0.3083 | Precision=0.5000 | F1=0.3814
K= 5 | chunks= 173 | Recall=0.4083 | Precision=0.4000 | F1=0.4041
K=10 | chunks= 173 | Recall=0.5333 | Precision=0.2500 | F1=0.3404

Testing chunk_size=1200, chunk_overlap=150
K= 1 | chunks= 131 | Recall=0.1458 | Precision=0.5000 | F1=0.2258
K= 3 | chunks= 131 | Recall=0.4792 | Precision=0.5000 | F1=0.4894
K= 5 | chunks= 131 | Recall=0.5625 | Precision=0.3500 | F1=0.4315
K=10 | chunks= 131 | Recall=0.7500 | Precision=0.2250 | F1=0.3462

Results saved to evaluation_results.csv
